# Bring your own mesh: the prebuilt mesh layout

The generated mesh layouts (`rectilinear`, `triangular`) place mesh nodes for you.
With `mesh_layout="prebuilt"` you instead supply **your own mesh node positions** --
for example ICON grid vertices, MPAS cell centres, or an observation-station network --
and `weather-model-graphs` builds the encode-process-decode graph around them.

The two-step mesh creation process still applies, with a twist:

1. **Coordinate creation** (`mesh_layout="prebuilt"`): your nodes are validated and
   passed through as an *edge-less node cloud* -- no adjacency is invented here.
2. **Connectivity creation** (`m2m_connectivity`): mesh edges are built directly from
   the node positions (`method="delaunay"`, the default) and directed with
   `len`/`vdiff` edge features.

```{note}
Currently the prebuilt layout supports **nodes-only** input: the mesh graph you
provide must not contain any edges. Support for user-provided edges is planned --
see the design discussion in
[issue #79](https://github.com/mllam/weather-model-graphs/issues/79).
```


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import weather_model_graphs as wmg

## The input contract

Provide a `networkx.Graph` where **every node** has:

- `pos`: `np.ndarray` of shape `(2,)` -- the node position. **These must be in the
  same coordinate system as the grid `coords`** you pass to
  `create_all_graph_components` (the library cannot check this for you!).
- `type`: the string `"mesh"`.
- `level`: an integer, **only** for hierarchical meshes (lowest value = finest
  level); either all nodes have one or none do.

Node positions must be unique. For the simplest case you can also pass a bare
`np.ndarray` of shape `[N, 2]` instead of a graph.


## Set up a fake grid and some mesh nodes

We create a regular grid of (x, y) coordinates (the locations of the input/output
data) and a set of irregular "station-like" mesh node positions inside the same
domain.


In [ ]:
xs, ys = np.meshgrid(np.linspace(0, 10, 32), np.linspace(0, 10, 32))
xy = np.stack([xs.flatten(), ys.flatten()], axis=-1)

rng = np.random.default_rng(seed=42)
mesh_xy = rng.random((40, 2)) * 10

my_mesh = nx.Graph()
for i, pos in enumerate(mesh_xy):
    my_mesh.add_node(i, pos=pos, type="mesh")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.3, label="grid points")
ax.scatter(mesh_xy[:, 0], mesh_xy[:, 1], s=40, marker="^", label="my mesh nodes")
ax.legend()
ax.set_title("User-provided mesh nodes over the data grid")

## Example 1 -- flat mesh from nodes only

With nodes-only input the connectivity step builds the mesh edges by Delaunay
triangulation of the node positions (`method="delaunay"` is the default, shown
here explicitly).


In [ ]:
graph_flat = wmg.create.create_all_graph_components(
    coords=xy,
    mesh_layout="prebuilt",
    mesh_layout_kwargs=dict(mesh_graph=my_mesh),
    m2m_connectivity="flat",
    m2m_connectivity_kwargs=dict(method="delaunay"),
    g2m_connectivity="nearest_neighbour",
    m2g_connectivity="nearest_neighbour",
)

m2m_flat = wmg.split_graph_by_edge_attribute(graph_flat, attr="component")["m2m"]

print(f"Mesh nodes : {m2m_flat.number_of_nodes()}")
print(f"Mesh edges : {m2m_flat.number_of_edges()}")

fig, ax = plt.subplots(figsize=(5, 5))
wmg.visualise.nx_draw_with_pos_and_attr(m2m_flat, ax=ax, node_size=30)
ax.set_title("Flat prebuilt mesh (Delaunay connectivity)")

The convenience form skips building the graph yourself -- pass the coordinate
array directly:


In [ ]:
graph_from_array = wmg.create.create_all_graph_components(
    coords=xy,
    mesh_layout="prebuilt",
    mesh_layout_kwargs=dict(mesh_graph=mesh_xy),
    m2m_connectivity="flat",
    g2m_connectivity="nearest_neighbour",
    m2g_connectivity="nearest_neighbour",
)
graph_from_array.number_of_nodes()

## Example 2 -- hierarchical mesh from `level` attributes

For a hierarchical mesh, give every node an integer `level` attribute (lowest
value = finest). Within each level the mesh edges are built per level
(`intra_level=dict(method="delaunay")`); between levels, `mesh_up`/`mesh_down`
edges are created by nearest-neighbour search (`inter_level`), exactly as for the
generated layouts.


In [ ]:
my_mesh_hier = nx.Graph()
for i, pos in enumerate(mesh_xy):
    my_mesh_hier.add_node(("fine", i), pos=pos, type="mesh", level=1)
coarse_xy = rng.random((8, 2)) * 10
for i, pos in enumerate(coarse_xy):
    my_mesh_hier.add_node(("coarse", i), pos=pos, type="mesh", level=2)

components = wmg.create.create_all_graph_components(
    coords=xy,
    mesh_layout="prebuilt",
    mesh_layout_kwargs=dict(mesh_graph=my_mesh_hier),
    m2m_connectivity="hierarchical",
    m2m_connectivity_kwargs=dict(
        intra_level=dict(method="delaunay"),
        inter_level=dict(pattern="nearest", k=1),
    ),
    g2m_connectivity="nearest_neighbour",
    m2g_connectivity="nearest_neighbour",
    return_components=True,
)

m2m_hier = components["m2m"]
for direction in ("same", "up", "down"):
    n = sum(
        1 for _, _, d in m2m_hier.edges(data=True) if d.get("direction") == direction
    )
    print(f"{direction:>5} edges: {n}")

fig, ax = plt.subplots(figsize=(5, 5))
wmg.visualise.nx_draw_with_pos_and_attr(m2m_hier, ax=ax, node_size=30)
ax.set_title("Hierarchical prebuilt mesh (2 levels)")

## Helpful validation errors

The input contract is checked up front, with errors that say what to fix:


In [ ]:
bad_mesh = nx.Graph()
bad_mesh.add_node(0, pos=np.array([1.0, 1.0]), type="mesh")
bad_mesh.add_node(1, pos=np.array([1.0, 1.0]), type="mesh")  # duplicate!

try:
    wmg.create.create_all_graph_components(
        coords=xy,
        mesh_layout="prebuilt",
        mesh_layout_kwargs=dict(mesh_graph=bad_mesh),
        m2m_connectivity="flat",
        g2m_connectivity="nearest_neighbour",
        m2g_connectivity="nearest_neighbour",
    )
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
with_edges = nx.Graph()
with_edges.add_node(0, pos=np.array([0.0, 0.0]), type="mesh")
with_edges.add_node(1, pos=np.array([5.0, 5.0]), type="mesh")
with_edges.add_edge(0, 1)  # user-provided edges: not yet supported

try:
    wmg.create.create_all_graph_components(
        coords=xy,
        mesh_layout="prebuilt",
        mesh_layout_kwargs=dict(mesh_graph=with_edges),
        m2m_connectivity="flat",
        g2m_connectivity="nearest_neighbour",
        m2g_connectivity="nearest_neighbour",
    )
except NotImplementedError as e:
    print(f"NotImplementedError: {e}")

## Loading real mesh sources

The library deliberately has no file-format support here -- whatever the source,
you load it into the node-cloud graph in a few lines. For example, station
locations from a CSV with columns `x`, `y` (already in the grid's coordinate
system):

```python
import pandas as pd

df = pd.read_csv("stations.csv")
my_mesh = nx.Graph()
for i, row in df.iterrows():
    my_mesh.add_node(i, pos=np.array([row.x, row.y]), type="mesh")
```

or ICON grid vertices from its NetCDF grid file (remember to transform lon/lat to
the grid's projected coordinate system first, e.g. with `pyproj`).
